# SafeX Sentiment Analysis on Client Feedback (Week 1 Task)

**Student Name:** Muhammad Abdullah  
**Email:** meharabdullah4337@gmail.com  
**University:** Lahore Garrison University (BSCS)  
**Track:** AI/ML - Group 3 (Male)  
**Project Objective:** Build a robust Natural Language Processing (NLP) pipeline using VADER and TextBlob to perform sentiment analysis on 30 client feedback records across SafeX cybersecurity services, generate visual metrics, and flag critical negative feedback for executive follow-up.

## 1. Imports and Setup

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP libraries
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Aesthetics
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.autolayout'] = True
print('All libraries imported successfully!')

## 2. Load and Explore Client Feedback Dataset

In [ ]:
dataset_path = os.path.join('dataset', 'client_feedback.csv')
df = pd.read_csv(dataset_path)
print(f'Total Reviews Loaded: {len(df)}')
df.head()

## 3. Hybrid Sentiment Analysis (VADER + TextBlob Engine)

In [ ]:
vader = SentimentIntensityAnalyzer()

def analyze_feedback(text):
    # Clean text
    clean = re.sub(r'\s+', ' ', str(text)).strip()
    
    # VADER Scoring
    v_scores = vader.polarity_scores(clean)
    compound = v_scores['compound']
    
    # TextBlob Scoring
    blob = TextBlob(clean)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    # Classification
    if compound >= 0.05:
        category = 'Positive'
    elif compound <= -0.05:
        category = 'Negative'
    else:
        category = 'Neutral'
        
    return pd.Series({
        'compound_score': round(compound, 4),
        'vader_pos': round(v_scores['pos'], 4),
        'vader_neu': round(v_scores['neu'], 4),
        'vader_neg': round(v_scores['neg'], 4),
        'textblob_polarity': round(polarity, 4),
        'textblob_subjectivity': round(subjectivity, 4),
        'category': category
    })

analyzed_df = df.copy()
analyzed_df[['compound_score', 'vader_pos', 'vader_neu', 'vader_neg', 'textblob_polarity', 'textblob_subjectivity', 'category']] = df['feedback_text'].apply(analyze_feedback)
analyzed_df.head()

## 4. Sentiment Distribution & Summary Statistics

In [ ]:
counts = analyzed_df['category'].value_counts()
total = len(analyzed_df)

print('='*50)
print('      SENTIMENT ANALYSIS SUMMARY REPORT')
print('='*50)
for cat, cnt in counts.items():
    pct = (cnt / total) * 100
    print(f'{cat:10s} : {cnt:2d} reviews ({pct:.1f}%)')
print('-'*50)
print(f'Average VADER Compound Score: {analyzed_df["compound_score"].mean():.4f}')
print('='*50)

## 5. Visualizations

In [ ]:
# Figure 1: Donut & Bar Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)
colors = {'Positive': '#10B981', 'Neutral': '#64748B', 'Negative': '#EF4444'}
chart_colors = [colors[c] for c in counts.index]

# Donut
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=chart_colors,
            wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
            textprops=dict(color='#1E293B', fontsize=12, weight='bold'))
axes[0].set_title('Sentiment Proportions', fontsize=14, weight='bold')

# Bar
bars = axes[1].bar(counts.index, counts.values, color=chart_colors, width=0.55, edgecolor='#1E293B')
axes[1].set_title('Review Count by Sentiment', fontsize=14, weight='bold')
axes[1].set_ylabel('Total Reviews')
for b in bars:
    axes[1].annotate(f'{b.get_height()}', (b.get_x() + b.get_width()/2, b.get_height()), xytext=(0, 4),
                     textcoords='offset points', ha='center', weight='bold')

plt.suptitle('SafeX Client Feedback Sentiment Distribution', fontsize=16, weight='bold')
plt.show()

In [ ]:
# Figure 2: Service-wise Sentiment Breakdown
service_ct = pd.crosstab(analyzed_df['service_type'], analyzed_df['category'])[['Positive', 'Neutral', 'Negative']]
fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
service_ct.plot(kind='barh', stacked=True, color=['#10B981', '#64748B', '#EF4444'], ax=ax, edgecolor='white')
ax.set_title('Sentiment Breakdown by SafeX Service Line', fontsize=14, weight='bold')
ax.set_xlabel('Number of Reviews')
plt.show()

## 6. Critical Alert System: Flag Top 3 Most Negative Comments

In [ ]:
neg_sorted = analyzed_df[analyzed_df['category'] == 'Negative'].sort_values(by=['compound_score', 'rating'], ascending=[True, True])
top_3_neg = neg_sorted.head(3)

print('='*70)
print('  🚨 TOP 3 CRITICAL NEGATIVE REVIEWS FLAGGED FOR IMMEDIATE ACTION')
print('='*70)

for i, (_, row) in enumerate(top_3_neg.iterrows(), 1):
    print(f'\n[ALERT #{i}] Client: {row["client_name"]} ({row["service_type"]})')
    print(f'  Rating: {row["rating"]}/5 | Compound Score: {row["compound_score"]}')
    print(f'  Comment: "{row["feedback_text"]}"')
    if 'communication' in row['feedback_text'].lower() or 'rude' in row['feedback_text'].lower():
        action = 'Escalate to Client Success Lead for direct manager apology and communication review.'
    elif 'failed' in row['feedback_text'].lower() or 'false positive' in row['feedback_text'].lower():
        action = 'Assign Senior Security Architect to re-evaluate technical deliverables and offer complimentary re-test.'
    else:
        action = 'Schedule urgent client retention sync within 24 hours.'
    print(f'  Action Plan: ➜ {action}')

## 7. Export Outputs

In [ ]:
os.makedirs('output', exist_ok=True)
analyzed_df.to_csv('output/analyzed_client_feedback.csv', index=False)
print('Exported processed dataset to output/analyzed_client_feedback.csv successfully!')